In [1]:
import pydata_google_auth
from google.cloud import bigquery

credentials = pydata_google_auth.get_user_credentials(
    ["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project="off-nutrition-pipeline", credentials=credentials)

table = client.get_table("off-nutrition-pipeline.bronze.produits")
print(table.num_rows, "lignes")

C:\Users\ayoub\OneDrive\Bureau\.venv\lib\site-packages\google\api_core\_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.5). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\ayoub\OneDrive\Bureau\.venv\lib\site-packages\google\api_core\_python_version_support.py:212: FutureWarning: You are using a non-supported Python version (3.10.5). Google will not post any further updates to google.cloud.bigquery supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.bigquery.
  warnings.warn(message, FutureWarning)


4882161 lignes


In [2]:
for champ in table.schema:
    print(champ.name, "|", champ.field_type, "|", champ.mode)

code | STRING | NULLABLE
product_name | RECORD | NULLABLE
brands | STRING | NULLABLE
brands_tags | RECORD | NULLABLE
categories_tags | RECORD | NULLABLE
countries_tags | RECORD | NULLABLE
nutriscore_grade | STRING | NULLABLE
nutriscore_score | INTEGER | NULLABLE
no_nutrition_data | BOOLEAN | NULLABLE
nutrition_data_per | STRING | NULLABLE
nutriments_json | BYTES | NULLABLE


In [3]:
sql = """
SELECT
  COUNT(*) AS total,
  COUNTIF(brands IS NULL OR TRIM(brands) = '') AS sans_marque,
  COUNTIF(nutriscore_grade IS NULL) AS grade_null,
  COUNTIF(nutriscore_grade = 'unknown') AS grade_unknown,
  COUNTIF(nutriscore_score IS NULL) AS score_null,
  COUNTIF(no_nutrition_data) AS sans_donnees_nutrition,
  COUNTIF(IFNULL(ARRAY_LENGTH(categories_tags.list), 0) = 0) AS sans_categorie,
  COUNTIF(EXISTS(SELECT 1 FROM UNNEST(categories_tags.list) t WHERE t.element = 'en:null')) AS categorie_en_null,
  COUNTIF(EXISTS(SELECT 1 FROM UNNEST(countries_tags.list) t WHERE t.element = 'en:world')) AS pays_en_world
FROM `off-nutrition-pipeline.bronze.produits`
"""
ligne = list(client.query(sql).result())[0]
for nom, valeur in ligne.items():
    print(nom, ":", valeur)

total : 4882161
sans_marque : 1753485
grade_null : 66506
grade_unknown : 3273668
score_null : 3449795
sans_donnees_nutrition : 5238
sans_categorie : 2920591
categorie_en_null : 1189
pays_en_world : 73290


In [4]:
sql = """
SELECT nutriscore_grade, COUNT(*) AS nb
FROM `off-nutrition-pipeline.bronze.produits`
GROUP BY nutriscore_grade
ORDER BY nb DESC
"""
for ligne in client.query(sql).result():
    print(ligne["nutriscore_grade"], ":", ligne["nb"])

unknown : 3273668
e : 400170
d : 358554
c : 299411
a : 209342
b : 164889
not-applicable : 109621
None : 66506


In [5]:
sql = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT code) AS codes_distincts,
  COUNTIF(code IS NULL OR TRIM(code) = '') AS code_vide,
  COUNTIF(IFNULL(ARRAY_LENGTH(product_name.list), 0) = 0) AS sans_nom
FROM `off-nutrition-pipeline.bronze.produits`
"""
ligne = list(client.query(sql).result())[0]
for nom, valeur in ligne.items():
    print(nom, ":", valeur)

total : 4882161
codes_distincts : 4882100
code_vide : 1
sans_nom : 298361


In [6]:
sql = """
SELECT code, COUNT(*) AS nb
FROM `off-nutrition-pipeline.bronze.produits`
GROUP BY code
HAVING COUNT(*) > 1
ORDER BY nb DESC
LIMIT 15
"""
for ligne in client.query(sql).result():
    print(repr(ligne["code"]), ":", ligne["nb"])

'2700963001059' : 2
'30383354190402' : 2
'516408508322002749' : 2
'5304067005431' : 2
'3700439700348' : 2
'2000000151669' : 2
'8008082903405' : 2
'7340011495437' : 2
'0059527401555' : 2
'0059527601559' : 2
'31843340000818' : 2
'6449027002163' : 2
'809507300522001927' : 2
'8026380707232' : 2
'2000000151670' : 2


In [7]:
sql = """
SELECT
  code,
  brands,
  product_name.list[SAFE_OFFSET(0)].element.text AS nom,
  nutriscore_grade,
  nutriscore_score,
  nutrition_data_per
FROM `off-nutrition-pipeline.bronze.produits`
WHERE code IN ('2700963001059', '3700439700348', '8008082903405')
ORDER BY code
"""
for ligne in client.query(sql).result():
    print(dict(ligne.items()))

{'code': '2700963001059', 'brands': 'Palais des Thés', 'nom': 'Jardin anglais', 'nutriscore_grade': 'unknown', 'nutriscore_score': None, 'nutrition_data_per': None}
{'code': '2700963001059', 'brands': '', 'nom': 'Jardin anglais', 'nutriscore_grade': None, 'nutriscore_score': None, 'nutrition_data_per': '100g'}
{'code': '3700439700348', 'brands': '', 'nom': None, 'nutriscore_grade': 'unknown', 'nutriscore_score': None, 'nutrition_data_per': '100g'}
{'code': '3700439700348', 'brands': '', 'nom': None, 'nutriscore_grade': None, 'nutriscore_score': None, 'nutrition_data_per': '100g'}
{'code': '8008082903405', 'brands': 'Cucina nostrana', 'nom': 'Melanzane alla parmigiana', 'nutriscore_grade': 'b', 'nutriscore_score': 1, 'nutrition_data_per': '100g'}
{'code': '8008082903405', 'brands': 'Cucina Nostrana', 'nom': 'Melanzane alla parmigiana', 'nutriscore_grade': 'b', 'nutriscore_score': 1, 'nutrition_data_per': None}


In [8]:
dataset = bigquery.Dataset("off-nutrition-pipeline.silver")
dataset.location = "EU"
client.create_dataset(dataset, exists_ok=True)
print("silver prête")

silver prête


In [10]:
t = client.get_table("off-nutrition-pipeline.bronze.produits")
print("bronze :", round(t.num_bytes / 1024**3, 2), "Gio")

for x in client.list_tables("off-nutrition-pipeline.silver"):
    print("dans silver :", x.table_id)
print("fin")

bronze : 9.38 Gio
fin


In [12]:
import os
from pathlib import Path

print("dossier courant :", os.getcwd())
for p in Path(r"C:\Users\ayoub\OneDrive\Bureau").rglob("bronze_off_v2.parquet"):
    print("trouvé :", p)
print("fin")

dossier courant : C:\Users\ayoub\OneDrive\Bureau\off_pipeline
trouvé : C:\Users\ayoub\OneDrive\Bureau\off_pipeline\data\bronze_off_v2.parquet
fin


In [13]:
options = bigquery.ParquetOptions()
options.enable_list_inference = True

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition="WRITE_TRUNCATE",
    parquet_options=options,
)

with open("data/bronze_off_v2.parquet", "rb") as f:
    job = client.load_table_from_file(
        f, "off-nutrition-pipeline.bronze.produits", job_config=job_config, location="EU"
    )
job.result()

t = client.get_table("off-nutrition-pipeline.bronze.produits")
print(t.num_rows, "lignes,", round(t.num_bytes / 1024**3, 2), "Gio")

4882161 lignes, 0.92 Gio


In [14]:
bornes = {
    "energie_kcal_100g": 900, "graisses_100g": 100, "graisses_saturees_100g": 100,
    "glucides_100g": 100, "sucres_100g": 100, "fibres_100g": 100,
    "proteines_100g": 100, "sel_100g": 100,
}
nutriments_sql = ",\n    ".join(
    f"IF({c} BETWEEN 0 AND {m}, {c}, NULL) AS {c}" for c, m in bornes.items()
)

sql = f"""
CREATE OR REPLACE TABLE `off-nutrition-pipeline.silver.produits` AS
WITH nettoye AS (
  SELECT
    code,
    NULLIF(TRIM(COALESCE(
      (SELECT t.text FROM UNNEST(product_name) t WHERE t.lang = 'main' LIMIT 1),
      product_name[SAFE_OFFSET(0)].text
    )), '') AS nom,
    NULLIF(LOWER(TRIM(SPLIT(brands, ',')[SAFE_OFFSET(0)])), '') AS marque,
    ARRAY(SELECT c FROM UNNEST(categories_tags) c
          WHERE c IS NOT NULL AND c != 'en:null') AS categories,
    ARRAY(SELECT p FROM UNNEST(countries_tags) p
          WHERE p IS NOT NULL AND p != 'en:world') AS pays,
    IF(nutriscore_grade IN ('a','b','c','d','e'), nutriscore_grade, NULL) AS nutriscore_lettre,
    nutriscore_score,
    IFNULL(nutriscore_grade, '') != 'not-applicable' AS nutriscore_applicable,
    no_nutrition_data,
    nutrition_data_per,
    {nutriments_sql}
  FROM `off-nutrition-pipeline.bronze.produits`
  WHERE code IS NOT NULL AND TRIM(code) != ''
)
SELECT * FROM nettoye
WHERE TRUE
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY code
  ORDER BY CAST(nom IS NOT NULL AS INT64) + CAST(marque IS NOT NULL AS INT64)
         + CAST(nutriscore_lettre IS NOT NULL AS INT64)
         + CAST(nutriscore_score IS NOT NULL AS INT64)
         + CAST(nutrition_data_per IS NOT NULL AS INT64) DESC
) = 1
"""
client.query(sql).result()

t = client.get_table("off-nutrition-pipeline.silver.produits")
print(t.num_rows, "lignes,", round(t.num_bytes / 1024**3, 2), "Gio")

4882099 lignes, 0.69 Gio


In [15]:
sql = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT code) AS codes_distincts,
  COUNTIF('en:null' IN UNNEST(categories)) AS categorie_en_null,
  COUNTIF('en:world' IN UNNEST(pays)) AS pays_en_world,
  COUNTIF(nutriscore_lettre IS NOT NULL) AS avec_lettre,
  COUNTIF(NOT nutriscore_applicable) AS nutriscore_non_applicable,
  COUNTIF(ARRAY_LENGTH(categories) > 0) AS avec_categorie,
  COUNTIF(marque IS NOT NULL) AS avec_marque,
  COUNTIF(sucres_100g IS NULL) AS sans_sucres,
  COUNTIF(energie_kcal_100g IS NULL) AS sans_energie
FROM `off-nutrition-pipeline.silver.produits`
"""
ligne = list(client.query(sql).result())[0]
for nom, valeur in ligne.items():
    print(nom, ":", valeur)

total : 4882099
codes_distincts : 4882099
categorie_en_null : 0
pays_en_world : 0
avec_lettre : 1432342
nutriscore_non_applicable : 109618
avec_categorie : 1960353
avec_marque : 3128524
sans_sucres : 1621674
sans_energie : 1397085


In [16]:
sql = """
SELECT ARRAY_LENGTH(categories) AS nb_categories, COUNT(*) AS nb_produits
FROM `off-nutrition-pipeline.silver.produits`
WHERE ARRAY_LENGTH(categories) > 0
GROUP BY nb_categories
ORDER BY nb_categories
LIMIT 12
"""
for l in client.query(sql).result():
    print(l["nb_categories"], "catégories :", l["nb_produits"], "produits")

print("--- trois exemples ---")
sql2 = """
SELECT code, categories
FROM `off-nutrition-pipeline.silver.produits`
WHERE ARRAY_LENGTH(categories) BETWEEN 4 AND 6
LIMIT 3
"""
for l in client.query(sql2).result():
    print(l["code"], list(l["categories"]))

1 catégories : 234851 produits
2 catégories : 138631 produits
3 catégories : 218994 produits
4 catégories : 248951 produits
5 catégories : 242812 produits
6 catégories : 201345 produits
7 catégories : 205134 produits
8 catégories : 141842 produits
9 catégories : 122472 produits
10 catégories : 69479 produits
11 catégories : 51395 produits
12 catégories : 31963 produits
--- trois exemples ---
5400119534437 ['en:meats-and-their-products', 'en:meats', 'en:chicken-and-its-products', 'en:poultries', 'en:chickens', 'en:chicken-thighs']
0609332827196 ['nl:beauty', 'nl:make-up', 'nl:wenkbrauw-make-up', 'nl:wenkbrauwpotlood']
06373035 ['en:sweeteners', 'en:syrups', 'en:simple-syrups', 'en:agave-syrups']


In [17]:
dataset = bigquery.Dataset("off-nutrition-pipeline.gold")
dataset.location = "EU"
client.create_dataset(dataset, exists_ok=True)

sql = """
CREATE OR REPLACE TABLE `off-nutrition-pipeline.gold.dim_pays` AS
SELECT
  ROW_NUMBER() OVER (ORDER BY nb_produits DESC, pays_code) AS pays_id,
  pays_code,
  INITCAP(REPLACE(REGEXP_REPLACE(pays_code, r'^[a-z]{2}:', ''), '-', ' ')) AS pays_nom,
  nb_produits,
  nb_produits >= 25000 AS pays_retenu
FROM (
  SELECT p AS pays_code, COUNT(*) AS nb_produits
  FROM `off-nutrition-pipeline.silver.produits`, UNNEST(pays) AS p
  GROUP BY p
)
"""
client.query(sql).result()

for l in client.query("""
    SELECT pays_id, pays_code, pays_nom, nb_produits, pays_retenu
    FROM `off-nutrition-pipeline.gold.dim_pays` ORDER BY pays_id LIMIT 8
""").result():
    print(l["pays_id"], l["pays_nom"], l["nb_produits"], l["pays_retenu"])

r = list(client.query("""
    SELECT COUNT(*) AS total, COUNTIF(pays_retenu) AS retenus
    FROM `off-nutrition-pipeline.gold.dim_pays`
""").result())[0]
print("pays au total :", r["total"], "| pays retenus (25 000 produits ou plus) :", r["retenus"])

1 France 1292664 True
2 United States 994448 True
3 Germany 436519 True
4 Spain 362433 True
5 Italy 273551 True
6 United Kingdom 199510 True
7 Canada 129817 True
8 Netherlands 115473 True
pays au total : 493 | pays retenus (25 000 produits ou plus) : 21


In [18]:
sql = """
CREATE OR REPLACE TABLE `off-nutrition-pipeline.gold.dim_marque` AS
SELECT
  ROW_NUMBER() OVER (ORDER BY nb_produits DESC, marque) AS marque_id,
  marque,
  nb_produits,
  nb_produits >= 30 AS marque_retenue
FROM (
  SELECT marque, COUNT(*) AS nb_produits
  FROM `off-nutrition-pipeline.silver.produits`
  WHERE marque IS NOT NULL
  GROUP BY marque
)
"""
client.query(sql).result()

for l in client.query("""
    SELECT marque_id, marque, nb_produits
    FROM `off-nutrition-pipeline.gold.dim_marque` ORDER BY marque_id LIMIT 8
""").result():
    print(l["marque_id"], l["marque"], l["nb_produits"])

r = list(client.query("""
    SELECT COUNT(*) AS total, COUNTIF(marque_retenue) AS retenues
    FROM `off-nutrition-pipeline.gold.dim_marque`
""").result())[0]
print("marques au total :", r["total"], "| marques retenues (30 produits ou plus) :", r["retenues"])

1 carrefour 24249
2 aldi 19518
3 coop 18840
4 lidl 18743
5 u 16722
6 auchan 14352
7 nestlé 14157
8 bonàrea 12251
marques au total : 392077 | marques retenues (30 produits ou plus) : 12916


In [19]:
sql = """
CREATE OR REPLACE TABLE `off-nutrition-pipeline.gold.dim_categorie` AS
WITH paires AS (
  SELECT
    categories[SAFE_OFFSET(0)] AS cat_generale,
    categories[SAFE_OFFSET(ARRAY_LENGTH(categories) - 1)] AS cat_precise
  FROM `off-nutrition-pipeline.silver.produits`
  WHERE ARRAY_LENGTH(categories) > 0
)
SELECT
  ROW_NUMBER() OVER (ORDER BY nb_produits DESC, cat_generale, cat_precise) AS categorie_id,
  cat_generale,
  cat_precise,
  INITCAP(REPLACE(REGEXP_REPLACE(cat_generale, r'^[a-z]{2}:', ''), '-', ' ')) AS generale_nom,
  INITCAP(REPLACE(REGEXP_REPLACE(cat_precise, r'^[a-z]{2}:', ''), '-', ' ')) AS precise_nom,
  STARTS_WITH(cat_generale, 'en:') AS normalisee,
  nb_produits
FROM (
  SELECT cat_generale, cat_precise, COUNT(*) AS nb_produits
  FROM paires
  GROUP BY cat_generale, cat_precise
)
"""
client.query(sql).result()

for l in client.query("""
    SELECT generale_nom, SUM(nb_produits) AS nb
    FROM `off-nutrition-pipeline.gold.dim_categorie`
    GROUP BY generale_nom ORDER BY nb DESC LIMIT 8
""").result():
    print(l["generale_nom"], ":", l["nb"])

r = list(client.query("""
    SELECT COUNT(*) AS lignes,
           COUNT(DISTINCT cat_generale) AS generales,
           ROUND(100 * SUM(IF(normalisee, nb_produits, 0)) / SUM(nb_produits), 1) AS pct_en
    FROM `off-nutrition-pipeline.gold.dim_categorie`
""").result())[0]
print("lignes :", r["lignes"], "| catégories générales :", r["generales"], "| % de produits avec catégorie en: :", r["pct_en"])

Plant Based Foods And Beverages : 529480
Snacks : 311157
Dairies : 168427
Beverages And Beverages Preparations : 167209
Meats And Their Products : 138769
Condiments : 89398
Meals : 66789
Seafood : 55829
lignes : 125217 | catégories générales : 44005 | % de produits avec catégorie en: : 96.8


In [20]:
sql = """
CREATE OR REPLACE TABLE `off-nutrition-pipeline.gold.dim_produit` AS
SELECT code, nom
FROM `off-nutrition-pipeline.silver.produits`
"""
client.query(sql).result()

sql = """
CREATE OR REPLACE TABLE `off-nutrition-pipeline.gold.fait_produit_pays` AS
SELECT
  s.code,
  p.pays_id,
  m.marque_id,
  c.categorie_id,
  s.nutriscore_lettre,
  s.nutriscore_score,
  s.nutriscore_applicable,
  s.energie_kcal_100g,
  s.graisses_100g,
  s.graisses_saturees_100g,
  s.glucides_100g,
  s.sucres_100g,
  s.fibres_100g,
  s.proteines_100g,
  s.sel_100g
FROM `off-nutrition-pipeline.silver.produits` s
LEFT JOIN UNNEST(s.pays) AS pc
LEFT JOIN `off-nutrition-pipeline.gold.dim_pays` p ON p.pays_code = pc
LEFT JOIN `off-nutrition-pipeline.gold.dim_marque` m ON m.marque = s.marque
LEFT JOIN `off-nutrition-pipeline.gold.dim_categorie` c
  ON c.cat_generale = s.categories[SAFE_OFFSET(0)]
 AND c.cat_precise = s.categories[SAFE_OFFSET(ARRAY_LENGTH(s.categories) - 1)]
"""
client.query(sql).result()

r = list(client.query("""
    SELECT COUNT(*) AS lignes,
           COUNT(DISTINCT code) AS produits_distincts,
           COUNTIF(pays_id IS NULL) AS sans_pays,
           COUNTIF(marque_id IS NOT NULL) AS avec_marque,
           COUNTIF(categorie_id IS NOT NULL) AS avec_categorie
    FROM `off-nutrition-pipeline.gold.fait_produit_pays`
""").result())[0]
for nom, valeur in r.items():
    print(nom, ":", valeur)

lignes : 5131127
produits_distincts : 4882099
sans_pays : 57331
avec_marque : 3358477
avec_categorie : 2168031
